In [1]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install parquet

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import parquet
from pathlib import Path

In [5]:
EIA = pd.read_csv("/home/danthebaguetteman/dev/CECS-399-499/local_data/silver/EIA_features_Final.csv")
EIA.columns

Index(['timestamp', 'demand_forecast_mwh', 'actual_demand_mwh',
       'net_generation_mwh', 'net_interchange_mwh', 'balance_error',
       'Percent Forecast Error', 'demand_ramp_rate_pct',
       'demand_rolling_mean_24h', 'demand_rolling_std_24h', 'demand_residual',
       'demand_zscore', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos'],
      dtype='str')

In [6]:
Eagle = pd.read_csv("/home/danthebaguetteman/dev/CECS-399-499/local_data/silver/eaglei_tennessee_cleaned_final.csv")
Eagle.columns

Index(['county', 'date', 'customers_out'], dtype='str')

In [53]:
tn_weather = pd.read_csv("../../local_data/silver/tn_weighted_weather_21_25.csv")
tn_weather.columns

Index(['timestamp', 'nashville_temperature_2m',
       'nashville_relative_humidity_2m', 'nashville_precipitation',
       'nashville_cloud_cover', 'nashville_wind_speed_10m',
       'nashville_shortwave_radiation', 'memphis_temperature_2m',
       'memphis_relative_humidity_2m', 'memphis_precipitation',
       'memphis_cloud_cover', 'memphis_wind_speed_10m',
       'memphis_shortwave_radiation', 'knoxville_temperature_2m',
       'knoxville_relative_humidity_2m', 'knoxville_precipitation',
       'knoxville_cloud_cover', 'knoxville_wind_speed_10m',
       'knoxville_shortwave_radiation', 'chattanooga_temperature_2m',
       'chattanooga_relative_humidity_2m', 'chattanooga_precipitation',
       'chattanooga_cloud_cover', 'chattanooga_wind_speed_10m',
       'chattanooga_shortwave_radiation', 'clarksville_temperature_2m',
       'clarksville_relative_humidity_2m', 'clarksville_precipitation',
       'clarksville_cloud_cover', 'clarksville_wind_speed_10m',
       'clarksville_shortwave_

In [8]:
doe = pd.read_csv("/home/danthebaguetteman/dev/CECS-399-499/local_data/silver/doe417_tennessee_cleaned_final.csv")
doe = doe.rename(columns={"Date Event Began":"date_start", "Time Event Began":"time_start", "Date of Restoration":"restoration_date",
                          "Area Affected":"area_affected", "Number of Customers Affected":"customers_affected",
                          "Alert Criteria":"alert_criteria", "Event Type":"event_type"})
doe.columns

Index(['Month', 'date_start', 'time_start', 'restoration_date',
       'Time of Restoration', 'area_affected', 'NERC Region', 'alert_criteria',
       'event_type', 'Demand Loss (MW)', 'customers_affected', 'source_year',
       'event_start_utc', 'event_end_utc'],
      dtype='str')

In [9]:
cities = pd.read_csv("/home/danthebaguetteman/dev/CECS-399-499/local_data/silver/tn_selected_city_population_weights.csv")
cities.columns

Index(['city', 'Geographic Area', 'population_2021', 'population_2022',
       'population_2023', 'population_2024', 'population_2025', 'weight_2021',
       'weight_2022', 'weight_2023', 'weight_2024', 'weight_2025'],
      dtype='str')

In [ ]:
def build_fact_energy_events(doe: pd.DataFrame) -> pd.DataFrame:
    df = doe.copy()

    df["demand_loss"] = pd.to_numeric(df["Demand Loss (MW)"], errors="coerce")
    df["number_affected"] = pd.to_numeric(
        df["customers_affected"], errors="coerce"
    )

    fact_energy_events = df[
        [   "event_start_utc",
            "event_end_utc",
            "alert_criteria",
            "event_type",
            "demand_loss",
            "number_affected",
        ]
    ].copy()
    return fact_energy_events


def write_parquet(df: pd.DataFrame, output_path: str) -> None:
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output, index=False, engine="pyarrow")
    print(f"Wrote {len(df):,} rows to {output}")

In [11]:
fact_energy_events = build_fact_energy_events(doe)

In [19]:
write_parquet(fact_energy_events, "../../local_data/gold/fact_energy_event.parquet")

Wrote 11 rows to ../../local_data/gold/fact_energy_event.parquet


In [47]:
pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 2.0 MB/s  0:00:23m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from pathlib import Path


def build_fact_outage_daily(Eagle: pd.DataFrame) -> pd.DataFrame:
    df = Eagle.copy()

    df = df.rename(columns={
        "date": "date",
        "county": "county",
        "customers_out": "customers_wo_power"
    })

    return df   

In [22]:
fact_outage_daily = build_fact_outage_daily(Eagle=Eagle)
write_parquet(fact_outage_daily, output_path="../../local_data/gold/fact_outage_daily.parquet")

Wrote 19,877 rows to ../../local_data/gold/fact_outage_daily.parquet


In [40]:
def build_fact_weather_city_hourly(tn_weather: pd.DataFrame) -> pd.DataFrame:
    df = tn_weather.copy()

    # Weather measures we want for each city
    measures = [
        "temperature_2m",
        "precipitation",
        "cloud_cover",
        "wind_speed_10m",
        "shortwave_radiation",
    ]

    # Detect available city prefixes from columns like nashville_temperature_2m
    city_measure_map = {}

    for col in df.columns:
        if col == "timestamp":
            continue
        for measure in measures:
            suffix = f"_{measure}"
            if col.endswith(suffix):
                city = col[: -len(suffix)]
                city_measure_map.setdefault(city, {})[measure] = col

    if not city_measure_map:
        raise ValueError("No city weather columns matching the expected patterns were found")

    rows = []

    for city, cols in city_measure_map.items():
        city_df = pd.DataFrame({
            "time_key": df["timestamp"],
            "city_key": city,
            "temperature": df[cols["temperature_2m"]],
            "precipitation": df[cols["precipitation"]],
            "cloud_cover": df[cols["cloud_cover"]],
            "wind_speed": df[cols["wind_speed_10m"]],
            "shortwave_radiation": df[cols["shortwave_radiation"]]
        })
        rows.append(city_df)

    fact_weather_city_hourly = pd.concat(rows, ignore_index=True)

    return fact_weather_city_hourly

In [24]:
fact_weather_city_hourly = build_fact_weather_city_hourly(tn_weather=tn_weather)

In [26]:
fact_weather_city_hourly.head()
fact_weather_city_hourly.dtypes

time_key               datetime64[us, UTC]
city_key                               str
temperature                        float64
precipitation                      float64
cloud_cover                        float64
wind_speed                         float64
shortwave_radiation                float64
dtype: object

In [27]:
write_parquet(fact_weather_city_hourly, output_path="../../local_data/gold/fact_weather_city_hourly.parquet")

Wrote 482,064 rows to ../../local_data/gold/fact_weather_city_hourly.parquet


In [28]:
def build_fact_energy_load_hourly(EIA: pd.DataFrame) -> pd.DataFrame:
    df = EIA.copy()

    df["source_id"] = "TVA"  # adjust if needed

    df = df.rename(columns={
        "timestamp": "time_key",
        "net_generation_mwh": "net_gen_mwh",
    })

    fact_energy_load_hourly = df[
        [
            "time_key",
            "source_id",
            "demand_forecast_mwh",
            "actual_demand_mwh",
            "net_gen_mwh",
        ]
    ].copy()

    return fact_energy_load_hourly

In [32]:
fact_energy_load_hourly = build_fact_energy_load_hourly(EIA=EIA)
fact_energy_load_hourly.head(10)

,time_key,source_id,demand_forecast_mwh,actual_demand_mwh,net_gen_mwh
0,2021-01-01 00:00:00+00:00,TVA,NaN,NaN,NaN
1,2021-01-01 01:00:00+00:00,TVA,NaN,NaN,NaN
2,2021-01-01 02:00:00+00:00,TVA,NaN,NaN,NaN
3,2021-01-01 03:00:00+00:00,TVA,NaN,NaN,NaN
4,2021-01-01 04:00:00+00:00,TVA,NaN,NaN,NaN
5,2021-01-01 05:00:00+00:00,TVA,14152.0,16270.0,16464.0
6,2021-01-01 06:00:00+00:00,TVA,13878.0,15673.0,15937.0
7,2021-01-01 07:00:00+00:00,TVA,14289.0,15105.0,15400.0
8,2021-01-01 08:00:00+00:00,TVA,13961.0,14618.0,14874.0
9,2021-01-01 09:00:00+00:00,TVA,13639.0,14299.0,14994.0


In [33]:
write_parquet(fact_energy_load_hourly, output_path="../../local_data/gold/fact_weather_city_hourly.parquet")

Wrote 43,824 rows to ../../local_data/gold/fact_weather_city_hourly.parquet


In [34]:
def build_fact_energy_features_hourly(EIA: pd.DataFrame) -> pd.DataFrame:
    df = EIA.copy()

    df["source_id"] = "TVA"   # temporary source key

    # Rename to target schema
    df = df.rename(columns={
        "timestamp": "time_key",
        "Percent Forecast Error": "forecast_error_pct",
        "demand_ramp_rate_pct": "demand_ramp_pct",
        "demand_rolling_mean_24h": "demand_rolling_mean",
        "demand_rolling_std_24h": "demand_rolling_std",
    })

    fact_energy_features_hourly = df[
        [
            "time_key",
            "source_id",
            "net_interchange_mwh",
            "balance_error",
            "forecast_error_pct",
            "demand_ramp_pct",
            "demand_rolling_mean",
            "demand_rolling_std",
            "demand_residual",
            "demand_zscore",
            "hour_sin",
            "hour_cos",
            "month_sin",
            "month_cos",
        ]
    ].copy()

    return fact_energy_features_hourly

In [37]:
fact_energy_features_hourly = build_fact_energy_features_hourly(EIA)
fact_energy_features_hourly.head(10)

,time_key,source_id,net_interchange_mwh,balance_error,forecast_error_pct,demand_ramp_pct,demand_rolling_mean,demand_rolling_std,demand_residual,demand_zscore,hour_sin,hour_cos,month_sin,month_cos
0,2021-01-01 00:00:00+00:00,TVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000e+00,0.5,0.866025
1,2021-01-01 01:00:00+00:00,TVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.258819,9.659258e-01,0.5,0.866025
2,2021-01-01 02:00:00+00:00,TVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,8.660254e-01,0.5,0.866025
3,2021-01-01 03:00:00+00:00,TVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.707107,7.071068e-01,0.5,0.866025
4,2021-01-01 04:00:00+00:00,TVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.866025,5.000000e-01,0.5,0.866025
5,2021-01-01 05:00:00+00:00,TVA,194.0,0.0,14.966083,NaN,16270.00000,NaN,0.000000,NaN,0.965926,2.588190e-01,0.5,0.866025
6,2021-01-01 06:00:00+00:00,TVA,264.0,0.0,12.934140,-3.669330,15971.50000,422.142748,-298.500000,-0.707107,1.000000,6.120000e-17,0.5,0.866025
7,2021-01-01 07:00:00+00:00,TVA,295.0,0.0,5.710687,-3.624067,15682.66667,582.560154,-577.666667,-0.991600,0.965926,-2.588190e-01,0.5,0.866025
8,2021-01-01 08:00:00+00:00,TVA,256.0,0.0,4.705967,-3.224098,15416.50000,713.883511,-798.500000,-1.118530,0.866025,-5.000000e-01,0.5,0.866025
9,2021-01-01 09:00:00+00:00,TVA,695.0,0.0,4.839064,-2.182241,15193.00000,794.973899,-894.000000,-1.124565,0.707107,-7.071068e-01,0.5,0.866025


In [38]:
write_parquet(fact_energy_features_hourly, output_path="../../local_data/gold/fact_energy_features_hourly.parquet")

Wrote 43,824 rows to ../../local_data/gold/fact_energy_features_hourly.parquet


In [46]:
import pandas as pd
from pathlib import Path

def build_dim_grid() -> pd.DataFrame:
    dim_grid = pd.DataFrame({
        "grid_id": ["TVA"],
        "grid_name": ["TVA"]
    })

    dim_grid["grid_id"] = dim_grid["grid_id"].astype("string")
    dim_grid["grid_name"] = dim_grid["grid_name"].astype("string")

    return dim_grid


def build_dim_time_hourly(EIA: pd.DataFrame = None, tn_weather: pd.DataFrame = None) -> pd.DataFrame:
    timestamp_series = []

    timestamp_series.append(EIA["timestamp"])
    timestamp_series.append(tn_weather["timestamp"])
    all_timestamps = pd.concat(timestamp_series, ignore_index=True).dropna().drop_duplicates()
    all_timestamps = pd.Series(all_timestamps).sort_values().reset_index(drop=True)

    dim_time_hourly = pd.DataFrame({
        "timestamp": all_timestamps
    })

    # Temporary key
    dim_time_hourly["time_id"] = dim_time_hourly["timestamp"]

    dim_time_hourly["hour_of_day"] = pd.to_datetime(dim_time_hourly["timestamp"]).dt.hour.astype("Int64")
    dim_time_hourly["day_of_week"] = pd.to_datetime(dim_time_hourly["timestamp"]).dt.dayofweek.astype("Int64")
    dim_time_hourly["is_weekend"] = pd.to_datetime(dim_time_hourly["day_of_week"]).isin([5, 6])
    dim_time_hourly["month"] = pd.to_datetime(dim_time_hourly["timestamp"]).dt.month.astype("Int64")
    dim_time_hourly["quarter"] = pd.to_datetime(dim_time_hourly["timestamp"]).dt.quarter.astype("Int64")
    dim_time_hourly["year"] = pd.to_datetime(dim_time_hourly["timestamp"]).dt.year.astype("Int64")

    dim_time_hourly = dim_time_hourly[
        [
            "time_id",
            "timestamp",
            "hour_of_day",
            "day_of_week",
            "is_weekend",
            "month",
            "quarter",
            "year",
        ]
    ].copy()

    return dim_time_hourly


def build_dim_city(cities: pd.DataFrame) -> pd.DataFrame:
    if "city" not in cities.columns:
        raise ValueError("cities dataset must contain a 'city' column")

    df = cities.copy()

    population_col = None
    for candidate in [
        "population_2025",
        "population_2024",
        "population_2023",
        "population_2022",
        "population_2021",
    ]:
        if candidate in df.columns:
            population_col = candidate
            break

    if population_col is None:
        df["population"] = pd.NA
    else:
        df["population"] = df[population_col]

    df["city_name"] = df["city"]
    df["city_id"] = df["city_name"].str.lower()
    df["state_code"] = df["Geographic Area"]

    df["latitude"] = pd.NA
    df["longitude"] = pd.NA

    dim_city = df[
        [
            "city_id",
            "city_name",
            "state_code",
            "population",
            "latitude",
            "longitude",
        ]
    ].drop_duplicates(subset=["city_id"]).reset_index(drop=True)

    return dim_city


def build_dim_county(Eagle: pd.DataFrame) -> pd.DataFrame:
    df = Eagle.copy()

    county_names = (
        df["county"]
        .astype("string")
        .str.strip()
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    dim_county = pd.DataFrame({
        "county_name": county_names
    })

    # Temporary key
    dim_county["county_id"] = dim_county["county_name"].str.lower()
    dim_county["state_code"] = "TN"
    dim_county["fips_code"] = pd.NA
    dim_county["population"] = pd.NA
    dim_county["latitude"] = pd.NA
    dim_county["longitude"] = pd.NA

    dim_county = dim_county[
        [
            "county_id",
            "county_name",
            "state_code",
            "fips_code",
            "population",
            "latitude",
            "longitude",
        ]
    ].copy()

    dim_county["county_id"] = dim_county["county_id"].astype("string")
    dim_county["county_name"] = dim_county["county_name"].astype("string")
    dim_county["state_code"] = dim_county["state_code"].astype("string")
    dim_county["fips_code"] = dim_county["fips_code"].astype("string")
    dim_county["population"] = pd.to_numeric(dim_county["population"], errors="coerce").astype("Int64")

    return dim_county

In [47]:
output_dir = Path("../../local_data/gold")

dim_grid = build_dim_grid()
dim_time_hourly = build_dim_time_hourly(EIA=EIA, tn_weather=tn_weather)
dim_city = build_dim_city(cities)
dim_county = build_dim_county(Eagle)

In [52]:
write_parquet(dim_grid, output_dir / "dim_grid.parquet")
write_parquet(dim_time_hourly, output_dir / "dim_time_hourly.parquet")
write_parquet(dim_city, output_dir / "dim_city.parquet")
write_parquet(dim_county, output_dir / "dim_county.parquet")

Wrote 1 rows to ../../local_data/gold/dim_grid.parquet
Wrote 43,824 rows to ../../local_data/gold/dim_time_hourly.parquet
Wrote 10 rows to ../../local_data/gold/dim_city.parquet
Wrote 89 rows to ../../local_data/gold/dim_county.parquet
